### LLM Basics

In [1]:
import os
from groq import Groq

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.environ.get("GROQ_API_KEY")
client = Groq(api_key=api_key)

### Tokens

Texts are broken down into tokens. A token is a basic unit of text that is processed by the model. 
LLMs have a maximum number of tokens that can be processed in a single request. The limit includes both
the input (prompt) and the output (generated text).

In [22]:
import tiktoken

# Load an encoding for a specific model
encoding = tiktoken.encoding_for_model("gpt-4o")

# Encode text into tokens
tokens = encoding.encode("I enjoy running into amoral people")
print(f"Encoded tokens: {tokens}")
print("Number of tokens:",len(tokens))

# Decode tokens back into text
decoded_text = encoding.decode(tokens)
print(f"Decoded text: {decoded_text}")
print("Number of words in text:",len(["I","enjoy","running","into","amoral","people"]))


Encoded tokens: [40, 4271, 6788, 1511, 939, 16882, 1665]
Number of tokens: 7
Decoded text: I enjoy running into amoral people
Number of words in text: 6


### Context Windows

A context window is the maximum amount of information an LLM can consider at a time. The model's performance
is directly impacted by the context window. A large context window would need more computation and memory, which can increase cost and processing time. With a small context window, the model may lsoe accuracy and coherence as it loses access to important context that would affect the output. Groq llama-3.3-70b-versatile has a context window of 131,072 tokens.

In [35]:
response1 = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Explain global warming."}])
usage1 = response1.usage
print("Explain global warming")
print("-----------------------------")
print(f"Prompt tokens:     {usage1.prompt_tokens}")
print(f"Completion tokens: {usage1.completion_tokens}")

response2 = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "What is the capital of India?"}])
usage2 = response2.usage
print("What is the capital of India?")
print("-----------------------------")
print(f"Prompt tokens:     {usage2.prompt_tokens}")
print(f"Completion tokens: {usage2.completion_tokens}")

Explain global warming
-----------------------------
Prompt tokens:     40
Completion tokens: 850
What is the capital of India?
-----------------------------
Prompt tokens:     42
Completion tokens: 9


In [40]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "My favourite ice cream flavour is chocolate chip cookie dough"},
              {"role": "assistant", "content": "Okay"},
              {"role": "user", "content": "What is my favourite ice cream flavour?"}])
print(response.choices[0].message.content)


Your favourite ice cream flavour is chocolate chip cookie dough.


### Temperature

The temperature parameter controls the randomness and creativity of the generated output. A low temperature would produce more predictable, focused and deterministic responses. A high temperature would result in more creative responses. This parameter is useful in cases where creative generation is valued, such as writing a story.

In [14]:
low_temp = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    temperature=0,
    max_tokens=150,
    messages=[{"role": "user", "content": "Write a slogan for a climate change awareness campaign"}],
)
print("Low Temperature")
print("----------------------------------")
print(low_temp.choices[0].message.content,"\n")
high_temp = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    temperature=1,
    max_tokens=150,
    messages=[{"role": "user", "content": "Write a slogan for a climate change awareness campaign"}],
)
print("High Temperature")
print("----------------------------------")
print(high_temp.choices[0].message.content)

Low Temperature
----------------------------------
"Rise for the Future: Every Small Step Today, A Cooler Tomorrow" 

High Temperature
----------------------------------
Here are a few slogan ideas for a climate change awareness campaign:

1. **"Rise for the Future: Act on Climate"** - Emphasizing the urgent need for collective action.
2. **"Cool the Planet, Not the Future"** - A playful way to highlight the importance of reducing greenhouse gas emissions.
3. **"Climate Change: It's Time to Change Our Ways"** - Encouraging individuals to re-examine their daily habits and make sustainable choices.
4. **"Small Steps Today, a Sustainable Tomorrow"** - Focusing on the positive impact of individual actions on the environment.
5. **"Unite for a Cooler Earth"** - Promoting global unity and cooperation in the fight against climate


### Max Tokens

The max tokens parameter limits the number of tokens that can be generated by the model. It controls the length of the generated output.

In [6]:
short_answer = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    max_tokens=15,
    messages=[{"role": "user", "content": "Explain global warming"}],
)
print("Max tokens: 15")
print("----------------------------------")
print(short_answer.choices[0].message.content,"\n")
long_answer = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    max_tokens=300,
    messages=[{"role": "user", "content": "Explain global warming"}],
)
print("Max tokens: 300")
print("----------------------------------")
print(long_answer.choices[0].message.content)

Max tokens: 15
----------------------------------
Global warming, also known as climate change, refers to the long-term rise 

Max tokens: 300
----------------------------------
**What is Global Warming?**

Global warming, also known as climate change, refers to the long-term rise in the average surface temperature of the Earth due to the increasing levels of greenhouse gases in the atmosphere. These gases, such as carbon dioxide (CO2), methane (CH4), and water vapor (H2O), trap heat from the sun and prevent it from being released back into space, leading to a warming effect on the planet.

**Causes of Global Warming**

The main causes of global warming can be attributed to human activities that release large amounts of greenhouse gases into the atmosphere. Some of the key contributors include:

1. **Burning of fossil fuels**: The extraction, transportation, and combustion of fossil fuels such as coal, oil, and gas release massive amounts of CO2 into the atmosphere.
2. **Deforestation*

### System Prompts vs User Prompts

System prompts define the AI's overall behavior and role.
User prompts provide specific instructions or questions to the AI for a particular task or interaction.

In [41]:
system_prompt = "You are a kind teacher explaining concepts to a 5 year old"
user_prompt = "What is global warming?"
response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "system", "content": system_prompt},
                 {"role": "user", "content": user_prompt}])
print(response.choices[0].message.content)

Oh boy, let's talk about something important. So, you know how sometimes it gets really hot outside and we have to drink lots of water to stay cool? Well, the Earth is like us, it can get hot too. And just like how we need to stay cool, the Earth needs to stay cool too.

There's something called global warming, which means the Earth is getting a little too hot. It's like when you leave your toy outside on a really sunny day and it gets too hot to touch. The Earth is getting hotter because of the bad things that people are doing, like throwing trash and yucky things into the air.

Imagine the Earth is wrapped in a big blanket, and that blanket is getting thicker and thicker. This blanket is called the atmosphere, and it's what keeps us warm. But sometimes, because of the bad things people do, the blanket gets too thick and it makes the Earth get too hot.

Now, you might wonder, what can we do to help the Earth? We can start by being kind to the Earth. We can plant more trees, not throw 

### Zero Shot Prompting vs Few Shot Prompting

Zero shot prompting: You ask the model to do the task without any examples

Few shot prompting: You show the model a few examples of input and output pairs to guide it
to give you the kind of answer you want

In [45]:
zero_shot = "Classify the sentiment of the following sentence: This song is amazing!"
zero_shot_ans = client.chat.completions.create(
    model="llama-3.3-70b-versatile", temperature=0,
    messages=[{"role": "user", "content": zero_shot}],
)
print("Zero-shot result:")
print(zero_shot_ans.choices[0].message.content)

Zero-shot result:
The sentiment of the sentence "This song is amazing!" is positive. The use of the word "amazing" indicates a strong enthusiasm and admiration for the song, expressing a very favorable opinion.


In [65]:
example1 = """Sentence: That movie was terrible.
Sentiment: Negative"""
example2 = """Sentence: The food was okay.
Sentiment: Neutral"""
example3 = """Sentence: I love your dress.
Sentiment: Positive"""

few_shot = f"""Classify the sentiment of the following sentence: This song is amazing! Here are some examples:
Example 1: {example1}, Example 2: {example2},Example 3: {example3}"""
few_shot_ans = client.chat.completions.create(
    model="llama-3.3-70b-versatile", temperature=0,
    messages=[{"role": "user", "content": few_shot}],
)
print("Few shot result:")
print(few_shot_ans.choices[0].message.content)

Few shot result:
Sentence: This song is amazing!
Sentiment: Positive


### Chain of Thought Prompting

COT prompting is a technique where the model generates step by step explanations before arriving at the final answer. It improves accuracy and makes the model more reliable. It is especially useful for math and logic questions.

In [61]:
prompt = """I have 26 cupcakes. Stacy takes 12 of them. John takes half of my remaining cupcakes and half 
of Stacy's cupcakes. How many cupcakes does we each have? Don't think step by step. Just give me a 
number as an answer."""
ans = client.chat.completions.create(
    model="llama-3.1-8b-instant", temperature=0,
    messages=[{"role": "user", "content": prompt}],
)
print(ans.choices[0].message.content)

Stacy: 12 
You: 26 - 12 = 14 
John: (14/2) + (12/2) = 8 + 6 = 14 
You: 14 - 8 = 6


In [62]:
prompt = """I have 26 cupcakes. Stacy takes 12 of them. John takes half of my remaining cupcakes and half 
of Stacy's cupcakes. How many cupcakes does we each have? Think step by step."""
ans = client.chat.completions.create(
    model="llama-3.1-8b-instant", temperature=0,
    messages=[{"role": "user", "content": prompt}],
)
print(ans.choices[0].message.content)

Let's break down the problem step by step.

Initially, you have 26 cupcakes.

1. Stacy takes 12 cupcakes from you. 
   Remaining cupcakes with you: 26 - 12 = 14
   Stacy has 12 cupcakes.

2. John takes half of your remaining cupcakes and half of Stacy's cupcakes.
   Half of your remaining cupcakes: 14 / 2 = 7
   Half of Stacy's cupcakes: 12 / 2 = 6
   John takes 7 cupcakes from you and 6 cupcakes from Stacy.

Now, let's calculate the number of cupcakes each person has:

- You have: 14 - 7 = 7 cupcakes
- Stacy has: 12 - 6 = 6 cupcakes
- John has: 7 (from you) + 6 (from Stacy) = 13 cupcakes
